In [2]:
"""
BCIC2A: 数据探索脚本
====================
"""

import numpy as np
import h5py
from scipy import signal as sp_signal
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/BCIC2A'

def load_h5(filepath):
    data_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            data_dict[key] = f[key][()]
    return data_dict

print("=" * 60)
print("BCIC2A 数据探索")
print("=" * 60)

train_data = load_h5(f'{DATA_DIR}/train.h5')
val_data = load_h5(f'{DATA_DIR}/val.h5')
test_data = load_h5(f'{DATA_DIR}/test_x_only.h5')

X_train = train_data['X']
y_train = train_data['y']
X_val = val_data['X']
y_val = val_data['y']
X_test = test_data['X']

print(f"\n[数据形状]")
print(f"  Train: {X_train.shape}, labels: {y_train.shape}")
print(f"  Val:   {X_val.shape}, labels: {y_val.shape}")
print(f"  Test:  {X_test.shape}")

print(f"\n[标签分布]")
for name, y in [('Train', y_train), ('Val', y_val)]:
    counts = np.bincount(y, minlength=4)
    print(f"  {name}: {counts}")
    print(f"    比例: {counts/counts.sum()}")

print(f"\n[信号统计 - 整体]")
for name, X in [('Train', X_train), ('Val', X_val)]:
    print(f"  {name}:")
    print(f"    mean={X.mean():.4f}, std={X.std():.4f}")
    print(f"    min={X.min():.2f}, max={X.max():.2f}")

print(f"\n[每个样本的均值/标准差 - 抽查100个]")
idx = np.random.choice(len(X_train), 100, replace=False)
sample_means = [X_train[i].mean() for i in idx]
sample_stds = [X_train[i].std() for i in idx]
print(f"  样本均值: mean={np.mean(sample_means):.4f}, std={np.std(sample_means):.4f}")
print(f"  样本标准差: mean={np.mean(sample_stds):.4f}, std={np.std(sample_stds):.4f}")

# 频谱分析
print(f"\n[功率谱密度分析 - C3/C4通道]")
fs = 200
f, psd_left = sp_signal.welch(X_train[y_train==0], fs=fs, nperseg=128, axis=-1)
f, psd_right = sp_signal.welch(X_train[y_train==1], fs=fs, nperseg=128, axis=-1)
f, psd_foot = sp_signal.welch(X_train[y_train==2], fs=fs, nperseg=128, axis=-1)
f, psd_tongue = sp_signal.welch(X_train[y_train==3], fs=fs, nperseg=128, axis=-1)

c3_idx = 7   # C3
c4_idx = 11  # C4

for band_name, (low, high) in [('delta',(1,4)), ('theta',(4,8)), ('mu',(8,12)), 
                                 ('beta',(12,30)), ('low_gamma',(30,50))]:
    band_mask = (f >= low) & (f < high)
    psd_left_band = psd_left[:, c3_idx, band_mask].mean()
    psd_right_band = psd_right[:, c3_idx, band_mask].mean()
    psd_foot_band = psd_foot[:, c3_idx, band_mask].mean()
    psd_tongue_band = psd_tongue[:, c3_idx, band_mask].mean()
    max_diff = max([psd_left_band, psd_right_band, psd_foot_band, psd_tongue_band]) - \
               min([psd_left_band, psd_right_band, psd_foot_band, psd_tongue_band])
    print(f"  {band_name:12s} | left={psd_left_band:.2e} right={psd_right_band:.2e} "
          f"foot={psd_foot_band:.2e} tongue={psd_tongue_band:.2e} | 极差={max_diff:.2e}")

# 标准化检测
print(f"\n[标准化检测]")
print(f"  全局均值: {X_train.mean():.4f}, 全局标准差: {X_train.std():.4f}")
channel_means = X_train.mean(axis=(0,2))
channel_stds = X_train.std(axis=(0,2))
print(f"  各通道均值范围: [{channel_means.min():.4f}, {channel_means.max():.4f}]")
print(f"  各通道标准差范围: [{channel_stds.min():.4f}, {channel_stds.max():.4f}]")

if abs(X_train.mean()) < 0.5 and abs(X_train.std() - 1.0) < 0.2:
    print("  ⚠️ 数据被全局z-score标准化了！")
elif channel_stds.min() > 0.8 and channel_stds.max() < 1.2:
    print("  ⚠️ 数据被per-channel标准化了！")
else:
    print("  ✓ 数据没有被标准化，保留了幅度信息")

print("\n" + "=" * 60)
print("探索完成")
print("=" * 60)

BCIC2A 数据探索

[数据形状]
  Train: (720, 22, 800), labels: (720,)
  Val:   (360, 22, 800), labels: (360,)
  Test:  (360, 22, 800)

[标签分布]
  Train: [180 180 180 180]
    比例: [0.25 0.25 0.25 0.25]
  Val: [90 90 90 90]
    比例: [0.25 0.25 0.25 0.25]

[信号统计 - 整体]
  Train:
    mean=0.0014, std=0.0130
    min=-0.12, max=0.12
  Val:
    mean=0.0013, std=0.0134
    min=-0.10, max=0.12

[每个样本的均值/标准差 - 抽查100个]
  样本均值: mean=0.0009, std=0.0054
  样本标准差: mean=0.0104, std=0.0026

[功率谱密度分析 - C3/C4通道]
  delta        | left=6.37e-06 right=6.55e-06 foot=7.02e-06 tongue=6.87e-06 | 极差=6.50e-07
  theta        | left=3.41e-06 right=3.50e-06 foot=3.83e-06 tongue=3.74e-06 | 极差=4.15e-07
  mu           | left=4.02e-06 right=4.29e-06 foot=5.07e-06 tongue=6.19e-06 | 极差=2.17e-06
  beta         | left=9.82e-07 right=1.04e-06 foot=1.13e-06 tongue=1.14e-06 | 极差=1.61e-07
  low_gamma    | left=2.42e-07 right=2.75e-07 foot=2.70e-07 tongue=2.32e-07 | 极差=4.22e-08

[标准化检测]
  全局均值: 0.0014, 全局标准差: 0.0130
  各通道均值范围: [0.0013, 0.0015]


In [3]:
"""
BCIC2A: CSP + SVM 基线
=======================
运动想象的经典方法。
"""

import numpy as np
import h5py
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from scipy import signal as sp_signal
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/BCIC2A'
N_CHANNELS = 22
N_CLASSES = 4

print("=" * 60)
print("BCIC2A: CSP + SVM 基线")
print("=" * 60)

# 加载数据
def load_h5(filepath):
    data_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            data_dict[key] = f[key][()]
    return data_dict

train_data = load_h5(f'{DATA_DIR}/train.h5')
val_data = load_h5(f'{DATA_DIR}/val.h5')
test_data = load_h5(f'{DATA_DIR}/test_x_only.h5')

X_train = train_data['X']
y_train = train_data['y']
X_val = val_data['X']
y_val = val_data['y']
X_test = test_data['X']

print(f"\n[数据]")
print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# 合并train+val
X_all = np.concatenate([X_train, X_val], axis=0)
y_all = np.concatenate([y_train, y_val], axis=0)

# =================== CSP实现 ===================

def compute_covariance(X):
    n_trials, n_channels, n_samples = X.shape
    covs = np.zeros((n_trials, n_channels, n_channels))
    for i in range(n_trials):
        covs[i] = np.cov(X[i]) / np.trace(np.cov(X[i]))
    return covs

def csp(X, y, n_filters=3):
    classes = np.unique(y)
    W_list = []
    for c in classes:
        X_c = X[y == c]
        X_other = X[y != c]
        cov_c = compute_covariance(X_c).mean(axis=0)
        cov_other = compute_covariance(X_other).mean(axis=0)
        eigvals, eigvecs = np.linalg.eig(np.linalg.inv(cov_other + 0.001 * np.eye(N_CHANNELS)) @ cov_c)
        idx = np.argsort(eigvals)[::-1]
        eigvecs = eigvecs[:, idx]
        W_list.append(eigvecs[:, :n_filters])
        W_list.append(eigvecs[:, -n_filters:])
    W = np.hstack(W_list)
    return W

def apply_csp(X, W):
    n_trials = X.shape[0]
    n_filters = W.shape[1]
    features = np.zeros((n_trials, n_filters))
    for i in range(n_trials):
        Z = W.T @ X[i]
        var = np.var(Z, axis=1)
        features[i] = np.log(var + 1e-10)
    return features

# =================== 方案1: CSP + SVM ===================

print("\n" + "="*60)
print("方案1: CSP + SVM")
print("="*60)

for n_filters in [2, 3, 4]:
    W = csp(X_all, y_all, n_filters=n_filters)
    feat_train = apply_csp(X_train, W)
    feat_val = apply_csp(X_val, W)
    
    clf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
    clf.fit(feat_train, y_train)
    val_pred = clf.predict(feat_val)
    val_acc = accuracy_score(y_val, val_pred)
    
    print(f"  n_filters={n_filters}: Val Accuracy = {val_acc:.4f}")

# =================== 方案2: 频带功率 + SVM ===================

print("\n" + "="*60)
print("方案2: 频带功率(mu+beta) + SVM")
print("="*60)

def extract_band_power(X, fs=200):
    bands = {'mu': (8, 12), 'beta': (12, 30), 'mu_beta': (8, 30)}
    features = []
    for trial in X:
        trial_feat = []
        for ch in range(X.shape[1]):
            f, psd = sp_signal.welch(trial[ch], fs=fs, nperseg=128)
            for (low, high) in bands.values():
                mask = (f >= low) & (f < high)
                trial_feat.append(np.mean(psd[mask]))
        features.append(trial_feat)
    return np.array(features)

feat_train_band = extract_band_power(X_train)
feat_val_band = extract_band_power(X_val)

clf_band = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
clf_band.fit(feat_train_band, y_train)
val_pred_band = clf_band.predict(feat_val_band)
val_acc_band = accuracy_score(y_val, val_pred_band)

print(f"  Val Accuracy = {val_acc_band:.4f}")

# =================== 方案3: 时域统计 + SVM ===================

print("\n" + "="*60)
print("方案3: 时域统计 + SVM")
print("="*60)

def extract_time_features(X):
    features = []
    for trial in X:
        feat = []
        for ch in range(X.shape[1]):
            feat.extend([trial[ch].mean(), trial[ch].std(),
                        np.max(trial[ch]), np.min(trial[ch])])
        features.append(feat)
    return np.array(features)

feat_train_time = extract_time_features(X_train)
feat_val_time = extract_time_features(X_val)

clf_time = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
clf_time.fit(feat_train_time, y_train)
val_pred_time = clf_time.predict(feat_val_time)
val_acc_time = accuracy_score(y_val, val_pred_time)

print(f"  Val Accuracy = {val_acc_time:.4f}")

# =================== 预测 ===================

print("\n" + "="*60)
print("预测test (用CSP方案)...")
print("="*60)

best_W = csp(X_all, y_all, n_filters=3)
feat_all = apply_csp(X_all, best_W)
feat_test = apply_csp(X_test, best_W)

final_clf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
final_clf.fit(feat_all, y_all)

test_pred = final_clf.predict(feat_test)

output_file = 'bcic2a_csp_predictions.txt'
np.savetxt(output_file, test_pred, fmt='%d')

print(f"\n✅ 已保存: {output_file}")
print(f"   left={np.sum(test_pred==0)}, right={np.sum(test_pred==1)}, "
      f"foot={np.sum(test_pred==2)}, tongue={np.sum(test_pred==3)}")

print("\n完成！")

BCIC2A: CSP + SVM 基线

[数据]
  Train: (720, 22, 800), Val: (360, 22, 800), Test: (360, 22, 800)

方案1: CSP + SVM
  n_filters=2: Val Accuracy = 0.3389
  n_filters=3: Val Accuracy = 0.3194
  n_filters=4: Val Accuracy = 0.3167

方案2: 频带功率(mu+beta) + SVM
  Val Accuracy = 0.3333

方案3: 时域统计 + SVM
  Val Accuracy = 0.3111

预测test (用CSP方案)...

✅ 已保存: bcic2a_csp_predictions.txt
   left=206, right=15, foot=49, tongue=90

完成！


In [4]:
"""
BCIC2A: EEGNet 时域基线
=======================
针对BCIC2A调参：
- 22通道
- 800时间点（4秒@200Hz）
- kernel_length=128（适配更长窗口）
- 4分类
"""

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/BCIC2A'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 150
LR = 0.001
PATIENCE = 25

print(f"使用设备: {DEVICE}")

# 加载数据
def load_h5(filepath):
    data_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            data_dict[key] = f[key][()]
    return data_dict

print("[步骤1] 加载数据...")
train_data = load_h5(f'{DATA_DIR}/train.h5')
val_data = load_h5(f'{DATA_DIR}/val.h5')
test_data = load_h5(f'{DATA_DIR}/test_x_only.h5')

X_train = train_data['X']
y_train = train_data['y']
X_val = val_data['X']
y_val = val_data['y']
X_test = test_data['X']

print(f"  Train: {X_train.shape}")
print(f"  Val:   {X_val.shape}")
print(f"  Test:  {X_test.shape}")

# =================== EEGNet（适配BCIC2A） ===================

class EEGNet_BCIC2A(nn.Module):
    def __init__(self, num_classes=4, channels=22, samples=800,
                 dropout_rate=0.5, kernel_length=128, F1=8, D=2):
        super().__init__()
        self.F1 = F1
        self.D = D
        self.F2 = F1 * D
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), 
                     padding=(0, kernel_length//2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, self.F2, (channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_rate)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(self.F2, self.F2, (1, 32), 
                     padding=(0, 16), groups=self.F2, bias=False),
            nn.Conv2d(self.F2, self.F2, 1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_rate)
        )
        
        # 自动计算flatten维度
        with torch.no_grad():
            dummy = torch.zeros(1, 1, channels, samples)
            x = self.conv1(dummy)
            x = self.conv2(x)
            self.flatten_dim = x.numel()
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, num_classes)
        )
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.classifier(x)
        return x

# 训练函数
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

# 主训练流程
print("\n[步骤2] 转为Tensor...")
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
X_val_t = torch.FloatTensor(X_val).unsqueeze(1)
y_val_t = torch.LongTensor(y_val)
X_test_t = torch.FloatTensor(X_test).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)

print("\n[步骤3] 构建 EEGNet...")
model = EEGNet_BCIC2A(num_classes=4, channels=22, samples=800,
                      dropout_rate=0.5, kernel_length=128, F1=8, D=2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {total_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8, verbose=True)

best_val_acc = 0
best_epoch = 0
no_improve = 0

print(f"\n[步骤4] 开始训练 ({EPOCHS}轮)...")
print("-" * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        
    if no_improve >= PATIENCE:
        print(f"\n⏹️ Early stopping (best={best_val_acc:.4f} at epoch {best_epoch})")
        break

model.load_state_dict(best_model_state)

# 评估
print(f"\n[步骤5] 最终评估 (Epoch {best_epoch})...")
print("-" * 60)
_, _, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
print(f"验证集准确率: {accuracy_score(val_labels, val_preds):.4f}")
print(classification_report(val_labels, val_preds, target_names=['left', 'right', 'foot', 'tongue']))
print("混淆矩阵:")
print(confusion_matrix(val_labels, val_preds))

# 预测
print("\n[步骤6] 生成测试集预测...")
model.eval()
with torch.no_grad():
    X_test_device = X_test_t.to(DEVICE)
    test_outputs = model(X_test_device)
    _, test_pred = torch.max(test_outputs, 1)
    test_pred = test_pred.cpu().numpy()

output_file = 'bcic2a_eegnet_predictions.txt'
np.savetxt(output_file, test_pred, fmt='%d')
print(f"\n✅ 已保存: {output_file}")
print(f"   left={np.sum(test_pred==0)}, right={np.sum(test_pred==1)}, "
      f"foot={np.sum(test_pred==2)}, tongue={np.sum(test_pred==3)}")

print("\n完成！")

使用设备: cuda
[步骤1] 加载数据...
  Train: (720, 22, 800)
  Val:   (360, 22, 800)
  Test:  (360, 22, 800)

[步骤2] 转为Tensor...

[步骤3] 构建 EEGNet...
模型总参数量: 3,828

[步骤4] 开始训练 (150轮)...
------------------------------------------------------------
Epoch   1/150 | Train: 0.2347 | Val: 0.2667
Epoch   5/150 | Train: 0.4542 | Val: 0.4250
Epoch  10/150 | Train: 0.5167 | Val: 0.4972
Epoch  15/150 | Train: 0.5722 | Val: 0.5083
Epoch  20/150 | Train: 0.5875 | Val: 0.5083
Epoch  25/150 | Train: 0.6250 | Val: 0.5167
Epoch 00027: reducing learning rate of group 0 to 5.0000e-04.
Epoch  30/150 | Train: 0.6389 | Val: 0.5028
Epoch  35/150 | Train: 0.6264 | Val: 0.5028
Epoch 00038: reducing learning rate of group 0 to 2.5000e-04.
Epoch  40/150 | Train: 0.6653 | Val: 0.4917
Epoch  45/150 | Train: 0.6708 | Val: 0.4889
Epoch 00047: reducing learning rate of group 0 to 1.2500e-04.
Epoch  50/150 | Train: 0.6931 | Val: 0.5139

⏹️ Early stopping (best=0.5361 at epoch 29)

[步骤5] 最终评估 (Epoch 29)...
--------------------------

In [1]:
"""
BCIC2A: ShallowConvNet + mu/beta带通滤波
===========================================
运动想象专用CNN + 频带预处理。
ShallowConvNet是BCI竞赛的经典架构，
第一层大kernel = 自动学习频带滤波器。
"""

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy import signal as sp_signal
import warnings
warnings.filterwarnings('ignore')

# =================== 固定随机种子 ===================
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("✓ 随机种子已固定 (seed=42)")

# =================== 配置 ===================
DATA_DIR = '/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/BCIC2A'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 150
LR = 0.001
PATIENCE = 25

# 带通滤波参数
LOW_FREQ = 8    # mu波段起点
HIGH_FREQ = 30  # beta波段终点
SAMPLING_RATE = 200

print(f"使用设备: {DEVICE}")
print(f"带通滤波: {LOW_FREQ}-{HIGH_FREQ}Hz")

# =================== 数据加载 ===================

def load_h5(filepath):
    data_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            data_dict[key] = f[key][()]
    return data_dict

print("[步骤1] 加载数据...")
train_data = load_h5(f'{DATA_DIR}/train.h5')
val_data = load_h5(f'{DATA_DIR}/val.h5')
test_data = load_h5(f'{DATA_DIR}/test_x_only.h5')

X_train = train_data['X']
y_train = train_data['y']
X_val = val_data['X']
y_val = val_data['y']
X_test = test_data['X']

print(f"  Train: {X_train.shape}")
print(f"  Val:   {X_val.shape}")
print(f"  Test:  {X_test.shape}")

# =================== 带通滤波预处理 ===================

def bandpass_filter(X, low, high, fs, order=4):
    """
    对每个trial的每个通道做带通滤波
    X: (trials, channels, samples)
    """
    nyq = fs / 2.0
    low_norm = low / nyq
    high_norm = high / nyq
    
    b, a = sp_signal.butter(order, [low_norm, high_norm], btype='band')
    
    X_filtered = np.zeros_like(X)
    for i in range(X.shape[0]):
        for ch in range(X.shape[1]):
            X_filtered[i, ch, :] = sp_signal.filtfilt(b, a, X[i, ch, :])
    
    return X_filtered

print(f"\n[步骤2] 带通滤波 {LOW_FREQ}-{HIGH_FREQ}Hz...")
X_train_f = bandpass_filter(X_train, LOW_FREQ, HIGH_FREQ, SAMPLING_RATE)
X_val_f = bandpass_filter(X_val, LOW_FREQ, HIGH_FREQ, SAMPLING_RATE)
X_test_f = bandpass_filter(X_test, LOW_FREQ, HIGH_FREQ, SAMPLING_RATE)

print(f"  滤波后: mean={X_train_f.mean():.6f}, std={X_train_f.std():.6f}")

# =================== ShallowConvNet ===================

class ShallowConvNet(nn.Module):
    """
    ShallowConvNet - BCI竞赛经典架构
    专为运动想象设计，第一层大kernel学频带滤波+空间滤波
    """
    def __init__(self, num_classes=4, channels=22, samples=800,
                 dropout_rate=0.5, F1=40, kernel_length=25):
        super().__init__()
        
        # Block 1: 时域卷积（学频带滤波）+ 空间卷积（学CSP-like投影）
        self.conv_block = nn.Sequential(
            # 时域卷积: 大kernel，覆盖约125ms (25点@200Hz)
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length//2), bias=False),
            nn.BatchNorm2d(F1),
            # 空间卷积: 跨所有通道
            nn.Conv2d(F1, F1, (channels, 1), bias=False),
            nn.BatchNorm2d(F1),
            nn.ELU(),
            nn.AvgPool2d((1, 75), stride=(1, 15)),  # 大池化，降采样
            nn.Dropout(dropout_rate)
        )
        
        # 自动计算flatten维度
        with torch.no_grad():
            dummy = torch.zeros(1, 1, channels, samples)
            x = self.conv_block(dummy)
            self.flatten_dim = x.numel()
        
        # 分类器
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, num_classes)
        )
        
    def forward(self, x):
        x = self.conv_block(x)
        x = self.classifier(x)
        return x

# =================== 训练函数 ===================

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

# =================== 主训练流程 ===================

print("\n[步骤3] 转为Tensor...")
X_train_t = torch.FloatTensor(X_train_f).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
X_val_t = torch.FloatTensor(X_val_f).unsqueeze(1)
y_val_t = torch.LongTensor(y_val)
X_test_t = torch.FloatTensor(X_test_f).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)

print("\n[步骤4] 构建 ShallowConvNet...")
model = ShallowConvNet(num_classes=4, channels=22, samples=800,
                       dropout_rate=0.5, F1=40, kernel_length=25).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {total_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8, verbose=True)

best_val_acc = 0
best_epoch = 0
no_improve = 0

print(f"\n[步骤5] 开始训练 ({EPOCHS}轮)...")
print("-" * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        
    if no_improve >= PATIENCE:
        print(f"\n⏹️ Early stopping (best={best_val_acc:.4f} at epoch {best_epoch})")
        break

model.load_state_dict(best_model_state)

# =================== 评估 ===================

print(f"\n[步骤6] 最终评估 (Epoch {best_epoch})...")
print("-" * 60)
_, _, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
print(f"验证集准确率: {accuracy_score(val_labels, val_preds):.4f}")
print(classification_report(val_labels, val_preds, target_names=['left', 'right', 'foot', 'tongue']))
print("混淆矩阵:")
print(confusion_matrix(val_labels, val_preds))

# =================== 预测 ===================

print("\n[步骤7] 生成测试集预测...")
model.eval()
with torch.no_grad():
    X_test_device = X_test_t.to(DEVICE)
    test_outputs = model(X_test_device)
    _, test_pred = torch.max(test_outputs, 1)
    test_pred = test_pred.cpu().numpy()

output_file = 'bcic2a_shallow_predictions.txt'
np.savetxt(output_file, test_pred, fmt='%d')
print(f"\n✅ 已保存: {output_file}")
print(f"   left={np.sum(test_pred==0)}, right={np.sum(test_pred==1)}, "
      f"foot={np.sum(test_pred==2)}, tongue={np.sum(test_pred==3)}")

print("\n" + "=" * 60)
print("完成！ShallowConvNet + 带通滤波～")
print("=" * 60)

✓ 随机种子已固定 (seed=42)
使用设备: cuda
带通滤波: 8-30Hz
[步骤1] 加载数据...
  Train: (720, 22, 800)
  Val:   (360, 22, 800)
  Test:  (360, 22, 800)

[步骤2] 带通滤波 8-30Hz...
  滤波后: mean=0.000001, std=0.006196

[步骤3] 转为Tensor...

[步骤4] 构建 ShallowConvNet...
模型总参数量: 44,204

[步骤5] 开始训练 (150轮)...
------------------------------------------------------------
Epoch   1/150 | Train: 0.3028 | Val: 0.2500
Epoch   5/150 | Train: 0.4319 | Val: 0.3361
Epoch  10/150 | Train: 0.5500 | Val: 0.3972
Epoch  15/150 | Train: 0.5819 | Val: 0.4194
Epoch  20/150 | Train: 0.6431 | Val: 0.4333
Epoch  25/150 | Train: 0.6889 | Val: 0.4389
Epoch  30/150 | Train: 0.6847 | Val: 0.4417
Epoch  35/150 | Train: 0.7167 | Val: 0.4500
Epoch  40/150 | Train: 0.7611 | Val: 0.4556
Epoch  45/150 | Train: 0.7528 | Val: 0.4694
Epoch  50/150 | Train: 0.7847 | Val: 0.4444
Epoch 00052: reducing learning rate of group 0 to 5.0000e-04.
Epoch  55/150 | Train: 0.8319 | Val: 0.4806
Epoch  60/150 | Train: 0.8708 | Val: 0.4833
Epoch  65/150 | Train: 0.8625 | Va

In [2]:
"""
BCIC2A: EEGNet + 合并数据 + 90/10分 + 数据增强
=================================================
"""

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# =================== 固定随机种子 ===================
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("✓ 随机种子已固定 (seed=42)")

# =================== 配置 ===================
DATA_DIR = '/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/BCIC2A'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 150
LR = 0.001
PATIENCE = 25
NOISE_STD = 0.005  # 数据增强噪声

print(f"使用设备: {DEVICE}")

# =================== 数据加载 ===================

def load_h5(filepath):
    data_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            data_dict[key] = f[key][()]
    return data_dict

print("[步骤1] 加载数据...")
train_data = load_h5(f'{DATA_DIR}/train.h5')
val_data = load_h5(f'{DATA_DIR}/val.h5')
test_data = load_h5(f'{DATA_DIR}/test_x_only.h5')

X_combined = np.concatenate([train_data['X'], val_data['X']], axis=0)
y_combined = np.concatenate([train_data['y'], val_data['y']], axis=0)
X_test = test_data['X']

print(f"  合并后总数据: {X_combined.shape}")

# 分出10%做验证
X_train, X_val, y_train, y_val = train_test_split(
    X_combined, y_combined, test_size=0.1,
    random_state=42, stratify=y_combined
)

print(f"  训练集: {X_train.shape}")
print(f"  验证集: {X_val.shape}")
print(f"  测试集: {X_test.shape}")

# =================== 数据增强 ===================

class EEGAugmentedDataset(torch.utils.data.Dataset):
    """带噪声增强的数据集"""
    def __init__(self, X, y, noise_std=0.005):
        self.X = torch.FloatTensor(X).unsqueeze(1)
        self.y = torch.LongTensor(y)
        self.noise_std = noise_std
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        y = self.y[idx]
        # 加噪声
        if self.noise_std > 0:
            noise = torch.randn_like(x) * self.noise_std
            x = x + noise
        return x, y

# =================== EEGNet ===================

class EEGNet_BCIC2A(nn.Module):
    def __init__(self, num_classes=4, channels=22, samples=800,
                 dropout_rate=0.5, kernel_length=128, F1=8, D=2):
        super().__init__()
        self.F1 = F1
        self.D = D
        self.F2 = F1 * D
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), 
                     padding=(0, kernel_length//2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, self.F2, (channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_rate)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(self.F2, self.F2, (1, 32), 
                     padding=(0, 16), groups=self.F2, bias=False),
            nn.Conv2d(self.F2, self.F2, 1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_rate)
        )
        
        with torch.no_grad():
            dummy = torch.zeros(1, 1, channels, samples)
            x = self.conv1(dummy)
            x = self.conv2(x)
            self.flatten_dim = x.numel()
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, num_classes)
        )
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.classifier(x)
        return x

# =================== 训练函数 ===================

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

# =================== 主流程 ===================

print("\n[步骤2] 构建 EEGNet...")
model = EEGNet_BCIC2A(num_classes=4, channels=22, samples=800,
                      dropout_rate=0.5, kernel_length=128, F1=8, D=2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {total_params:,}")

# 数据加载器
train_dataset = EEGAugmentedDataset(X_train, y_train, noise_std=NOISE_STD)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

X_val_t = torch.FloatTensor(X_val).unsqueeze(1)
y_val_t = torch.LongTensor(y_val)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)  # 更强的L2
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8, verbose=True)

best_val_acc = 0
best_epoch = 0
no_improve = 0

print(f"\n[步骤3] 开始训练 ({EPOCHS}轮)...")
print(f"  噪声增强: std={NOISE_STD}")
print(f"  weight_decay=5e-4 (更强的正则化)")
print("-" * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        
    if no_improve >= PATIENCE:
        print(f"\n⏹️ Early stopping (best={best_val_acc:.4f} at epoch {best_epoch})")
        break

model.load_state_dict(best_model_state)

# =================== 评估 ===================

print(f"\n[步骤4] 最终评估 (Epoch {best_epoch})...")
print("-" * 60)
_, _, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
print(f"验证集准确率: {accuracy_score(val_labels, val_preds):.4f}")
print(classification_report(val_labels, val_preds, target_names=['left', 'right', 'foot', 'tongue']))
print("混淆矩阵:")
print(confusion_matrix(val_labels, val_preds))

# =================== 预测 ===================

print("\n[步骤5] 生成测试集预测...")
model.eval()
with torch.no_grad():
    X_test_device = torch.FloatTensor(X_test).unsqueeze(1).to(DEVICE)
    test_outputs = model(X_test_device)
    _, test_pred = torch.max(test_outputs, 1)
    test_pred = test_pred.cpu().numpy()

output_file = 'bcic2a_eegnet_merge_predictions.txt'
np.savetxt(output_file, test_pred, fmt='%d')
print(f"\n✅ 已保存: {output_file}")
print(f"   left={np.sum(test_pred==0)}, right={np.sum(test_pred==1)}, "
      f"foot={np.sum(test_pred==2)}, tongue={np.sum(test_pred==3)}")

print("\n" + "=" * 60)
print("完成！")
print("=" * 60)

✓ 随机种子已固定 (seed=42)
使用设备: cuda
[步骤1] 加载数据...
  合并后总数据: (1080, 22, 800)
  训练集: (972, 22, 800)
  验证集: (108, 22, 800)
  测试集: (360, 22, 800)

[步骤2] 构建 EEGNet...
模型总参数量: 3,828

[步骤3] 开始训练 (150轮)...
  噪声增强: std=0.005
  weight_decay=5e-4 (更强的正则化)
------------------------------------------------------------
Epoch   1/150 | Train: 0.2747 | Val: 0.2593
Epoch   5/150 | Train: 0.4321 | Val: 0.4630
Epoch  10/150 | Train: 0.5010 | Val: 0.4630
Epoch  15/150 | Train: 0.5412 | Val: 0.5093
Epoch  20/150 | Train: 0.5473 | Val: 0.4444
Epoch 00024: reducing learning rate of group 0 to 5.0000e-04.
Epoch  25/150 | Train: 0.5586 | Val: 0.4259
Epoch  30/150 | Train: 0.5689 | Val: 0.5000
Epoch 00035: reducing learning rate of group 0 to 2.5000e-04.
Epoch  35/150 | Train: 0.5967 | Val: 0.4907
Epoch  40/150 | Train: 0.5813 | Val: 0.4907
Epoch 00044: reducing learning rate of group 0 to 1.2500e-04.
Epoch  45/150 | Train: 0.5844 | Val: 0.5000
Epoch  50/150 | Train: 0.5977 | Val: 0.5000

⏹️ Early stopping (best=0.52